# Chapter 2 — Working with Tools

## Setup Instructions

To ensure you have the required dependencies to run this notebook, you'll need to have our `llm-agents-from-scratch` framework installed on the running Jupyter kernel. To do this, you can launch this notebook with the following command while within the project's root directory:

```sh
uv run --with jupyter jupyter lab
```

Alternatively, if you just want to use the published version of `llm-agents-from-scratch` without local development, you can install it from PyPi by uncommenting the cell below.

<!-- provider-note:start -->
## Setting the backbone LLM of your agent

These notebooks run on **Ollama by default**, the setup the book teaches.
If you do nothing, nothing changes: `make_llm()` starts a local Ollama
service when one isn't already running.

To use OpenAI or Anthropic instead:

1. Install the extra: `uv sync --extra openai` or `uv sync --extra anthropic`.
2. Export `OPENAI_API_KEY` or `ANTHROPIC_API_KEY` before launching Jupyter.
3. Set `LLM_PROVIDER=openai` (or `anthropic`), or pass `provider="openai"`
   to `make_llm()`. A key on its own never switches providers, so one
   exported for unrelated work cannot reroute you off the Ollama path.

The switch applies wherever a notebook builds its LLM with `make_llm()`.
A notebook that constructs `OllamaLLM` directly stays on Ollama regardless
of these settings.

If you opted in but forgot to export the key, you will be prompted for it.
That is the safe path on hosted kernels, and it keeps the key out of the
saved notebook. Setting `OLLAMA_API_KEY` alone routes Ollama to Ollama
Cloud.

**Caveat:** the examples are tuned for qwen3. Output on gpt-5 or Claude
will differ from what is printed in the book, and prompt-sensitive
examples (the ch09 evaluator pattern, ch08 supervised trajectories) may
behave noticeably differently.
<!-- provider-note:end -->

In [2]:
# Uncomment the line below to install `llm-agents-from-scratch` from PyPi
!pip install llm-agents-from-scratch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.4/118.4 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 276.0/276.0 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 368.7/368.7 kB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 406.5/406.5 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.8/129.8 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 68.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.1/324.1 kB 21.1 MB/s eta 0:00:00


## Examples

### Example 1: Creating a `ToolCall` object

In [3]:
from llm_agents_from_scratch.data_structures.tool import ToolCall

croissant_tool_call = ToolCall(
    tool_name="web-search-tool",
    arguments={
        "query": "Croissant bakeries in New York City and their prices.",
    },
)

In [ ]:
croissant_tool_call

ToolCall(id_='77148f15-74d5-4b38-9e9f-d39620bb7031', tool_name='web-search-tool', arguments={'query': 'Croissant bakeries in New York City and their prices.'})

### Example 2: Creating a `ToolCallResult` object

In [4]:
from llm_agents_from_scratch.data_structures.tool import ToolCallResult

result = ToolCallResult(
    tool_call_id=croissant_tool_call.id_,
    content={
        "search_results": {
            "hits": [
                {"data": "..."},
            ],
        },
    },
)

In [5]:
result

ToolCallResult(tool_call_id='107c5eb8-167a-46b7-8549-e55da488c77f', content={'search_results': {'hits': [{'data': '...'}]}}, error=False)

### Example 3: The Hailstone Tool

In [6]:
from typing import Any

from llm_agents_from_scratch.base.tool import BaseTool
from llm_agents_from_scratch.data_structures.tool import (
    ToolCall,
    ToolCallResult,
)


class Hailstone(BaseTool):  # A
    @property
    def name(self) -> str:
        return "hailstone"

    @property
    def description(self) -> str:
        return "A tool that performs a Hailstone step on a given input number."

    @property
    def parameters_json_schema(self) -> dict[str, Any]:
        """JSON Schema for tool parameters."""
        return {
            "type": "object",
            "properties": {
                "x": {
                    "type": "number",
                    "description": "The input number.",
                },
            },
            "required": ["x"],
        }

    def __call__(
        self,
        tool_call: ToolCall,
        *args: Any,
        **kwargs: Any,
    ) -> ToolCallResult:
        """Execute the tool call."""
        x = tool_call.arguments.get("x")
        if x % 2 == 0:
            result = x // 2
        else:
            result = (x * 3) + 1

        return ToolCallResult(
            tool_call_id=tool_call.id_,
            content=result,
            error=False,
        )

Invoking the Hailstone tool:

In [7]:
hailstone_tool = Hailstone()

tool_call = ToolCall(
    tool_name="hailstone",
    arguments={"x": 3},
)

tool_call_result = hailstone_tool(tool_call)
print(tool_call_result)

tool_call_id='cf3c73bc-19cb-49f4-9d4a-20cfe9ee100d' content=10 error=False


### Example 4: The Hailstone step function (to be used in `SimpleFunctionTool`)

In [8]:
def hailstone_step_func(x: int) -> int:
    """Performs a single step of the Hailstone sequence."""
    if x % 2 == 0:
        return x // 2
    return 3 * x + 1

### Example 5: Hailstone step function as a `SimpleFunctionTool`

In [9]:
from llm_agents_from_scratch.tools.simple_function import SimpleFunctionTool

# convert our Python function to a BaseTool
hailstone_tool = SimpleFunctionTool(hailstone_step_func)

print(hailstone_tool.name)
print(hailstone_tool.description)
print(hailstone_tool.parameters_json_schema)

hailstone_step_func
Performs a single step of the Hailstone sequence.
{'type': 'object', 'properties': {'x': {'type': 'number'}}, 'required': ['x']}


### Example 6: Running the previous Hailstone `tool_call` with the alternative `SimpleFunctionTool` implementation

In [10]:
from llm_agents_from_scratch.data_structures import ToolCall

tool_call = ToolCall(
    tool_name="hailstone_fn",
    arguments={"x": 3},
)

res = hailstone_tool(tool_call)

In [11]:
res

ToolCallResult(tool_call_id='7b016f4f-67de-4ba7-9a03-1907b6cdbe69', content='10', error=False)

### Example 7: Example usage of `PydanticFunctionTool`

In [12]:
from pydantic import BaseModel


class MyFuncParams(BaseModel):
    x: int


def my_func(params: MyFuncParams) -> int:
    print(params.x)

In [13]:
from llm_agents_from_scratch.tools.pydantic_function import PydanticFunctionTool

tool = PydanticFunctionTool(my_func)

In [14]:
tool

# Exercise 1

In [41]:
import asyncio

In [43]:
from typing import Any
from llm_agents_from_scratch.base.tool import AsyncBaseTool
from llm_agents_from_scratch.data_structures.tool import ToolCall, ToolCallResult

class Async_Haliston_Tool(AsyncBaseTool):

    @property
    def name(self) -> str:
        return "async_hailstone"

    @property
    def description(self) -> str:
        return "A tool that performs a Hailstone step on a given input number."

    @property
    def parameters_json_schema(self) -> dict[str, Any]:
        return {
            "type": "object",
            "properties": {
                "x": {
                    "type": "number",
                    "description": "The input number.",
                },
            },
            "required": ["x"],
        }

    async def __call__(
        self,
        tool_call: ToolCall,
        *args: Any,
        **kwargs: Any,
    ) -> ToolCallResult:
        """Execute the async tool call."""

        await asyncio.sleep(1)

        x = tool_call.arguments.get("x")
        if x % 2 == 0:
            result = x // 2
        else:
            result = (x * 3) + 1

        return ToolCallResult(
            tool_call_id=tool_call.id_,
            content=result,
            error=False,
        )

In [48]:
aync_haliston_tool_class = Async_Haliston_Tool()

tool_call = ToolCall(
    tool_name = "async_halistone",
    arguments = {"x":3}
)


res = await aync_haliston_tool_class(tool_call)
print(res)

tool_call_id='cc469f5b-e500-4188-9002-a94167526fd6' content=10 error=False


# Exercise 2

In [29]:
from llm_agents_from_scratch.tools.simple_function import AsyncSimpleFunctionTool

In [45]:
async def hailstone_step_func(x: int) -> int:
    """Performs a async single step of the Hailstone sequence."""
    await asyncio.sleep(1)
    if x % 2 == 0:
        return x // 2
    return 3 * x + 1

In [49]:
async_haliston_tool_simp = AsyncSimpleFunctionTool(func = hailstone_step_func)

print(async_haliston_tool_simp.name)
print(async_haliston_tool_simp.description)
print(async_haliston_tool_simp.parameters_json_schema)

hailstone_step_func
Performs a async single step of the Hailstone sequence.
{'type': 'object', 'properties': {'x': {'type': 'number'}}, 'required': ['x']}


In [51]:
tool_call = ToolCall(
    tool_name = "async_halistone",
    arguments = {"x":3}
)
res_simp =  await async_haliston_tool_simp(tool_call)
res_class = await aync_haliston_tool_class(tool_call)
print(res_simp)
print(res_class)

tool_call_id='ad114e75-3af1-4741-9066-8f206998b3c2' content='10' error=False
tool_call_id='ad114e75-3af1-4741-9066-8f206998b3c2' content=10 error=False


In [53]:
print("Class Tool Result Type:", type(res_class.content), "| Value:", res_class.content)
print("Simple Tool Result Type:", type(res_simp.content), "| Value:", res_simp.content)

# Verification by casting both to integers to handle type serialization differences
assert int(res_class.content) == int(res_simp.content)
print("✅ Success: Both implementations returned identical numeric outputs!")

Class Tool Result Type: <class 'int'> | Value: 10
Simple Tool Result Type: <class 'str'> | Value: 10
✅ Success: Both implementations returned identical numeric outputs!


# Excerise 3


In [54]:
from pydantic import BaseModel, Field, field_validator
from llm_agents_from_scratch.tools import PydanticFunctionTool

In [55]:
class Hailstone_Params(BaseModel):
  x : int = Field(description = "The input number.")

In [56]:
def halistone_func_step(params: Hailstone_Params) -> int:
  x = params.x
  if x % 2 == 0:
    return x // 2
  return 3 * x + 1

In [57]:
pydanic_haliston_tool = PydanticFunctionTool(halistone_func_step)

In [58]:
tool_call = ToolCall(
    tool_name = "async_halistone",
    arguments = {"x":3}
)
res = pydanic_haliston_tool(tool_call)
print(res)


tool_call_id='5b806edb-fcc6-480e-b3fb-3138b6c2d7ee' content='10' error=False
